In [83]:
import pandas as pd
import csv
import os
from collections import Counter
#import openpyxl
import re
import json

In [84]:
'''
read csv file load dataframe into memory
'''
path_to_file = '../hammoq_metadata_files/products.csv'

In [85]:
'''
if any element from keywords_list available in string_
return true
'''
def any_found(string_,keywords_list):
    return any(string in string_ for string in keywords_list)

In [ ]:
'''
read products.csv file and fetch all matching cases for given keyword
'''

In [92]:
keyword = 'top'

'''
keywords to include to include categories
'''
keywords_to_include = ['women','woman']
'''
keywords to exclude categories 
'''
keywords_to_exclude = ['shoe','sneaker','loafer','boot']

'''
columns from products dataframe to process
'''

imp_col=['productString','title','category','shortDescription','keywords']


def fetch_keyword_matching_products (products_df = products_df,
                                     keyword=keyword,imp_col=imp_col,
                                     keywords_to_include=keywords_to_include,
                                     keywords_to_exclude=keywords_to_exclude):
    
    prod_ids = []
    image_ids = []
    output=[]
    counter=0
    print(f'processing')
    for product in products_df['products']:
        for prod in json.loads(product):
            tmp_keys = prod.keys()
            '''
            pick common columns among imp_cols and prod.keys
            '''
            fnl_cols = set(imp_col).intersection(set(tmp_keys))
            fnl_cols = list(fnl_cols)
            string_ =''
            '''
            for all those columns pick strings and build one string
            '''
            for i in fnl_cols:
                string_ = string_+str(prod[i])
            string_ = string_.lower().replace('%20',' ').replace('%0a',' ').replace('%3a',' ')
            try:
                if keyword in string_ and any_found(string_,keywords_to_include) and not any_found(string_,keywords_to_exclude):

                    #prod_ids.append(prod['_id']['$oid'])
                    #print('done')
                    images = prod['images']
                    fnl_imgs = [v for k,v in images.items()]
                    #image_ids.append(fnl_imgs)
                    output.append({'prod_id':prod['_id']['$oid'],'images':fnl_imgs,'string':string_})
                    counter+=1

            except:
                print('skipped')
                pass

    print(f'match of {keyword} found in {counter} products')
    return output


In [ ]:
'''
Input products.csv is big, around 4.6 gb
we need to parse in chunks
'''

big_data = pd.read_csv(path_to_file,chunksize=100)
data = []
while (1) :

    try:
        products_df = next(big_data,'end')
        for datum in fetch_keyword_matching_products(products_df=products_df):
            data.append(datum)
        
    except Exception as e:
        print(e)
        break

In [ ]:
pd.DataFrame(data)

In [95]:
pd.DataFrame(data).to_csv(f'../women_tops/historical_data_analysis/get_historical_images/output/csv/big_data_women_{keyword}_to_get.csv')